In [8]:
import os
import json
import time
import pandas as pd
import re
from openai import OpenAI
from tqdm import tqdm

# ========== 1. 配置 ==========
API_KEY = ""
MODEL_NAME = "qwen-plus"
INPUT_FILE = "gold_standard1.xlsx"          # 黄金样本文件
OUTPUT_FILE = "gold_standard_predicted1.csv" # 预测结果输出
CACHE_FILE = "label_extraction_cache.json"

# 测试模式：可限制处理条数
START_ROW_INDEX = None      # 从第几行开始，None表示从头
NUM_SAMPLES = None          # 处理多少条，None表示全部
IGNORE_CACHE_FOR_TEST = False

# ========== 2. 初始化客户端 ==========
client = OpenAI(
    api_key=API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# ========== 3. 构建优化后的 System Prompt（16标签提取，含完整规则和示例） ==========
SYSTEM_PROMPT_16LABELS = """
你是严谨的诈骗分类标注助手。请严格按以下规则判断每个标签是否存在（1=存在，0=不存在）。输出简体中文，不主观发挥。

# 一、最高优先级规则（必须优先执行）

## 1. NS（信息不足）强制规则
若文本满足以下任一条件，必须设 NS=1，且所有其他标签为 0：
① 纯疑问句且无具体诈骗手法描述；
② 少于 30 字且无明确 G/L/R/C 元素；
③ 仅为新闻标题或链接分享，无正文；
④ 内容与诈骗完全无关。

## 2. 严禁过度解读（防止 G2/L1/L2/R1 过标）
仅当文本中明确出现**骗子对受害者的第一人称引诱或威胁**时才标注对应标签。普通社交求助、天气讨论、科普警示中的“关系”或“威胁”词汇不得标注。

## 3. 界面操控触发词（防止 C 类漏标）
出现以下表述时必须检查对应 C 标签：
① “点击链接” → C1 误点击
② “输入密码/验证码/身份证照片” → C2 误输入
③ “仿冒/假网站/相似域名” → C3 误识别
④ “下载 APP/开启权限/扫描二维码” → C4 误触发
⑤ “远程协助/屏幕共享/TeamViewer” → C5 远程控制

## 4. OT（其他叙事）使用限制
OT 是极罕见兜底标签，仅在确认诈骗且故事完全不属于 G/L/R/C 时使用（例如纯情感故事，无利益、威胁、冒充、界面）。若犹豫，优先归入 NS 或具体 G/L/R。OT 不能与任何 G/L/R/C 共存。

# 二、常规标签定义
[后续保留原有的 G/L/R/C 定义和输出格式...]

## 输出格式
严格按模板输出，标签顺序固定为：
G1,G2,G3,L1,L2,L3,R1,R2,R3,C1,C2,C3,C4,C5,OT,NS

示例1（纯财富诱饵）：
样本ID：1
标签向量：1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

示例2（冒充+威胁+误点击）：
样本ID：2
标签向量：0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0

示例3（信息不足）：
样本ID：3
标签向量：0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1

请严格遵循以上规则，不要添加额外解释。
"""

# ========== 4. 构建 User Prompt（只放可变内容） ==========
def build_user_prompt(query, idx):
    return f"""样本ID：{idx}
文本：{query}"""

# ========== 5. 调用 Qwen 提取标签向量 ==========
def call_qwen_extract(query, idx):
    user_prompt = build_user_prompt(query, idx)
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_16LABELS},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0,
                timeout=60
            )
            raw_output = response.choices[0].message.content
            lines = raw_output.strip().split('\n')
            vector_str = None
            for line in lines:
                if '标签向量：' in line:
                    vector_str = line.split('标签向量：')[1].strip()
                    break
            if vector_str is None:
                for line in lines:
                    if re.match(r'^[0-1](,[0-1]){15}$', line.strip()):
                        vector_str = line.strip()
                        break
            if vector_str is not None:
                return vector_str
            print(f"警告：未找到标签向量，原始输出：{raw_output}")
            return ""
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"\n样本ID {idx} 重试 {max_retries} 次仍失败: {e}")
                return ""
            time.sleep(2 ** attempt)
    return ""

# ========== 6. 主程序 ==========
def main():
    # 加载缓存
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r', encoding='utf-8') as f:
            cache = json.load(f)
        print(f"已加载缓存 {len(cache)} 条")
    else:
        cache = {}
    
    # 读取黄金样本文件
    if INPUT_FILE.endswith('.xlsx'):
        df = pd.read_excel(INPUT_FILE)
    else:
        df = pd.read_csv(INPUT_FILE)
    print(f"原始数据共 {len(df)} 行")
    
    # 确保有文本列（常见列名：content / text / 文本 / 内容）
    text_col = None
    for col in ['alltext']:
        if col in df.columns:
            text_col = col
            break
    if text_col is None:
        raise KeyError("表格中缺少文本列，请添加 content/text/文本/内容 列")
    
    # 确定ID列
    if '样本ID' not in df.columns:
        df['样本ID'] = df.index
        print("未找到 '样本ID' 列，将使用行索引作为样本ID")
    
    # 确定处理范围
    if START_ROW_INDEX is not None:
        start_idx = START_ROW_INDEX
    else:
        start_idx = 0
    if NUM_SAMPLES is not None:
        end_idx = min(start_idx + NUM_SAMPLES, len(df))
    else:
        end_idx = len(df)
    df_subset = df.iloc[start_idx:end_idx].copy()
    print(f"处理范围：行索引 {start_idx} 到 {end_idx-1}，共 {len(df_subset)} 条")
    
    # 准备结果列
    label_cols = ['G1','G2','G3','L1','L2','L3','R1','R2','R3',
                  'C1','C2','C3','C4','C5','OT','NS']
    for col in label_cols:
        if col not in df.columns:
            df[col] = None   # 新建预测列
    df['pred_vector'] = ""
    
    # 循环处理
    success_count = 0
    save_interval = 50
    for idx_in_subset, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="提取标签"):
        sample_id = str(row['样本ID'])
        
        # 检查缓存
        if not IGNORE_CACHE_FOR_TEST and sample_id in cache:
            vector_str = cache[sample_id]
            df.at[idx_in_subset, 'pred_vector'] = vector_str
            if vector_str:
                # 将向量拆分到各列
                parts = vector_str.split(',')
                if len(parts) == 16:
                    for i, col in enumerate(label_cols):
                        df.at[idx_in_subset, col] = int(parts[i])
                success_count += 1
            continue
        
        # 处理新数据
        query = row[text_col]
        if pd.isna(query) or str(query).strip() == "":
            vector_str = ""
        else:
            vector_str = call_qwen_extract(query, sample_id)
        
        # 保存结果
        df.at[idx_in_subset, 'pred_vector'] = vector_str
        cache[sample_id] = vector_str
        if vector_str:
            parts = vector_str.split(',')
            if len(parts) == 16:
                for i, col in enumerate(label_cols):
                    df.at[idx_in_subset, col] = int(parts[i])
            success_count += 1
        
        # 实时保存缓存
        with open(CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
        
        # 周期性保存 CSV
        if (idx_in_subset - start_idx + 1) % save_interval == 0:
            df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
            print(f"\n[进度] 已处理 {(idx_in_subset - start_idx + 1)} 条，成功 {success_count} 条，中间结果已保存")
    
    # 最终保存
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    total_processed = len(df_subset)
    print(f"\n处理完成！共处理 {total_processed} 条，成功获取非空向量 {success_count} 条")
    print(f"结果已保存至 {OUTPUT_FILE}")
    print(f"缓存文件：{CACHE_FILE}")

if __name__ == "__main__":
    main()

原始数据共 463 行
处理范围：行索引 0 到 462，共 463 条


提取标签:  11%|███▍                            | 50/463 [01:59<07:58,  1.16s/it]


[进度] 已处理 50 条，成功 50 条，中间结果已保存


提取标签:  22%|██████▋                        | 100/463 [03:25<08:38,  1.43s/it]


[进度] 已处理 100 条，成功 100 条，中间结果已保存


...


[进度] 已处理 400 条，成功 386 条，中间结果已保存


提取标签:  97%|██████████████████████████████▏| 450/463 [14:02<00:15,  1.18s/it]

警告：未找到标签向量，原始输出：G2,G1,0,L1,0,0,R1,0,0,0,0,0,0,0,0,0

[进度] 已处理 450 条，成功 435 条，中间结果已保存


提取标签: 100%|███████████████████████████████| 463/463 [14:48<00:00,  1.92s/it]


样本ID 462 重试 3 次仍失败: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-a7baa719-668d-9f92-b48a-8affcb1c2e83', 'request_id': 'a7baa719-668d-9f92-b48a-8affcb1c2e83'}

处理完成！共处理 463 条，成功获取非空向量 447 条
结果已保存至 gold_standard_predicted1.csv
缓存文件：label_extraction_cache.json
